In [4]:
import os


import torch
import torch.nn as nn
import torch.optim as optim

from PIL import Image
from torchvision import models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, ConcatDataset

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


### 1. Подготовка данных ###

In [6]:
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])

Dataset_Path = r"C:\Users\Alina\Desktop\ПРОЕКТ_Monkey\Monkey_project\dataset"

Dataset = datasets.ImageFolder(root = Dataset_Path, transform = transform)

Dataset_size = len(Dataset)

#разделение данные
train_size = int(0.7 * Dataset_size)
val_size = int(0.15 * Dataset_size)
test_size = Dataset_size - train_size - val_size

train_data, val_data, test_data = random_split(Dataset, [train_size, val_size, test_size])
print(Dataset.classes)

#Dataloader
batch = 16
train_loader = DataLoader(train_data, batch_size=batch, shuffle = True)
val_loader = DataLoader(val_data, batch_size=batch, shuffle = False)
test_loader = DataLoader(test_data, batch_size=batch, shuffle = False)

['Ateles_geoffroyi', 'Cebus_imitator', 'Gorilla_beringei', 'Macaca_fascicularis', 'Pan_troglodytes', 'Pongo_pygmaeus']


In [7]:
print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))
print("Dataset:", len(Dataset))

Train: 634
Val: 136
Test: 137
Dataset: 907


In [9]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)
print(labels)

torch.Size([16, 3, 224, 224])
torch.Size([16])
tensor([5, 1, 2, 2, 3, 5, 3, 1, 3, 2, 3, 2, 2, 4, 4, 5])


### Модель ###

In [10]:
model = models.resnet18(pretrained=True)

C:\Users\Alina\miniconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Alina\miniconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [11]:
print(model.fc) #последний слой имеет 1000 признаков, нам столько не нужно

Linear(in_features=512, out_features=1000, bias=True)


In [12]:
#заменяем последний слой
num_classes = 6
model.fc = nn.Linear(in_features=512, out_features=num_classes)
print(model.fc)

Linear(in_features=512, out_features=6, bias=True)


In [13]:
#замораживаем модель, теперь веса не буду обновляться
for param in model.parameters():
    param.requires_grad = False
#разморажмваем последний слой, чтобы обучалась только эта часть
for param in model.fc.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(model.fc.parameters(), lr = 0.001)
criterion = nn.CrossEntropyLoss()

num_epochs = 5

for epoch in range(num_epochs):
    
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_accuracy = 100 * correct/ total  
    
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:}")
    print(f"Val Accuracy: {val_accuracy:}")

Epoch [1/5]
Train Loss: 1.5051753520965576
Val Accuracy: 75.73529411764706
Epoch [2/5]
Train Loss: 1.0238656193017959
Val Accuracy: 80.88235294117646
Epoch [3/5]
Train Loss: 0.75184545814991
Val Accuracy: 86.02941176470588
Epoch [4/5]
Train Loss: 0.6203545905649662
Val Accuracy: 83.82352941176471
Epoch [5/5]
Train Loss: 0.5494728296995163
Val Accuracy: 82.3529411764706


In [14]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_accuracy = 100 * correct/ total
print(f"Test Accuracy: {test_accuracy:}")

Test Accuracy: 82.48175182481752


In [16]:
torch.save(model.state_dict(), "best_model.pth")

### Embedding ###

In [27]:
model.eval()

#заменила последний слой,чтобы получить вектор признаков
model.fc = nn.Identity()
model = model.to(device)

#Подготовка данных
full_data = ConcatDataset([train_data, val_data, test_data])
full_dataLoader = DataLoader(full_data, batch_size = batch, shuffle = False)

#сохраняю все пути к изображению
image_path = []

for subset in [train_data, val_data, test_data]:
    for idx in subset.indices:
        full_path, _ = Dataset.samples[idx]

       # находим позицию слова "dataset"
        split_index = full_path.lower().find("dataset")

        # берём только от dataset и дальше
        rel_path = full_path[split_index:]

        # заменяем слеши на /
        rel_path = rel_path.replace("\\", "/")

        image_path.append(rel_path)
    
#генерация эмбеддингов
embedding = []

with torch.no_grad():
    for images, _ in full_dataLoader:
        images = images.to(device)

        output = model(images) #[batch_size, 512]

        embedding.append(output.cpu())

#обьединяем все батчи в один
all_embeddings = torch.cat(embedding, dim = 0)

#создаем словарь: путь - эмбеддинг
image_to_embedding = {image_path[i]: all_embeddings[i] for i in range(len(image_path))}

print("Количество изображений:", len(image_path))
print("Размер одного embedding:", all_embeddings[0].shape)
print(image_path[0])

Количество изображений: 907
Размер одного embedding: torch.Size([512])
dataset/Ateles_geoffroyi/img_019.jpg


### Поиск похожих ###

In [28]:
model.eval()

model.fc = nn.Identity()
model = model.to(device)

#загружаю новую картинку и применяю те же трансформации, что и в датасете
def load_image(file, transform, device):
    image = Image.open(file).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)
    return image

#вычисляю эмбеддинг для новой картинки
def emb(model, image):
    with torch.no_grad():
        emb = model(image)
        emb = emb.cpu()
    return emb

#ищем 5 похожих изображений
def find(emb_new, embeddings, image_path):
    """
    emb_new — embedding новой картинки [1,512]
    embeddings — [N,512]
    image_path — пути к изображениям
    """
    emb_norm = embeddings / embeddings.norm(dim = 1, keepdim = True)
    emb_new_norm = emb_new / emb_new.norm(dim = 1, keepdim = True)

    #косинусное сходство
    similarity = torch.mm(emb_new_norm, emb_norm.t()).squeeze(0) #размер [N] столбец
    
    #беру топ 5 индексов с наиб схожестью
    top5_ind = torch.topk(similarity, 5).indices.tolist() #tensor([values, ind]) ->  tensor[ind] -> list(ind)
    #пути к этим изображениям
    top5_paths = [image_path[i] for i in top5_ind]
    
    return top5_paths

In [29]:
torch.save((all_embeddings, image_path), "embeddings.pt")


In [24]:
#Проверка
new_image_path = r"C:\Users\Alina\Desktop\ПРОЕКТ_Monkey\Monkey_project\img_goril.jpg"

image_tensor = load_image(new_image_path, transform, device)

emb_new = emb(model, image_tensor)


top = find(emb_new, all_embeddings, image_path)

print("5 похожих изображений:")
for p in top:
    print(p)

5 похожих изображений:
C:\Users\Alina\Desktop\ПРОЕКТ_Monkey\Monkey_project\dataset\Pan_troglodytes\img_183.jpg
C:\Users\Alina\Desktop\ПРОЕКТ_Monkey\Monkey_project\dataset\Gorilla_beringei\img_060.jpg
C:\Users\Alina\Desktop\ПРОЕКТ_Monkey\Monkey_project\dataset\Pan_troglodytes\img_067.jpg
C:\Users\Alina\Desktop\ПРОЕКТ_Monkey\Monkey_project\dataset\Gorilla_beringei\img_175.jpg
C:\Users\Alina\Desktop\ПРОЕКТ_Monkey\Monkey_project\dataset\Gorilla_beringei\img_139.jpg
